## What you must deliver:

At least 3 new features engineered from F1 domain knowledge. Each must use only pre-race information (no leakage). 
Each must include a 1–2 sentence justification: why this feature should help predict your target.

Feature types must include at least 2 of the following: lag feature (e.g., previous race position), rolling aggregate (e.g., average finish over last N races), interaction feature (e.g., driver × circuit type), or categorical encoding (e.g., constructor tier based on historical performance).

One simple model trained on the new features: Logistic Regression or Decision Tree (sklearn). No ensembles, no deep learning—the point is features, not model complexity.

Temporal validation maintained from Lab 1: walk-forward CV or season-based holdout. Same split boundaries.

comparison_table.md with side-by-side results: Lab 1 baselines vs. Lab 2 model. Same primary metric. Same validation. Honest interpretation of the gap (or lack thereof).

Error analysis: identify your model’s top-3 failure modes. For each: which races/drivers does it get wrong? Why? What would you try next?

Leakage guard checklist applied to all new features. Document the result in the notebook.

PROMPTS.md if you used AI tools. Follow the 6-field format: Context, Prompt, Output, Validation, Adaptations, Final Decision.

RANDOM_SEED = 414 in all random_state= arguments.

## Feature Engineering Guidance
You do not need sklearn Pipelines for this lab (that’s Monday March 23 content). Use pandas operations to create features. Here are examples relevant to F1:

Example features (adapt to your prediction target)
Lag feature: df['prev_race_position'] — driver’s finishing position in the immediately previous race.

Rolling aggregate: df['avg_position_last_3'] = df.groupby('driver_id')['position'].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))

Interaction: df['driver_at_circuit'] = df['driver_id'].astype(str) + '_' + df['circuit_id'].astype(str) — historical driver performance at specific circuits.

Categorical encoding: Constructor tier (top/mid/back) based on previous season standings.

Leakage warning
Every feature must use only information available before the race you are predicting. If a feature uses race-day results (finishing position, pit stop count, lap times), it is leaking the future. Apply the 10-item leakage guard checklist to every new feature. The .shift(1) pattern is your friend.

## Setup

In [1]:
# Load and prepare 2022-2024 race-result data from FastF1/Ergast for EDA.
import fastf1
import pandas as pd
import os

# ── YOUR TURN ──────────────────────────────────────────────────────────────
# Task: Implement majority-class and domain heuristic baselines on YOUR data.
# 
# Step 1: Load your Lab 1 data (copy your data loading code here)

# Configure local cache so repeated API reads are fast and reproducible.
cache_path = os.path.abspath("fastf1_cache")
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Create Ergast interface and iterate all race weekends in each season.
erg = fastf1.ergast.Ergast()

years = [2020, 2021, 2022, 2023, 2024]
results_list = []

for year in years:
    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():
        # Ignore testing sessions because they are not part of race prediction.
        if event['EventFormat'] == 'testing':
            continue

        rnd = int(event['RoundNumber'])

        try:
            res = erg.get_race_results(season=year, round=rnd)
            race = res.content[0]

            # Keep temporal/event metadata for later grouping and split logic.
            race['year'] = year
            race['round'] = rnd
            race['circuit'] = event['EventName']

            results_list.append(race)

        except Exception as e:
            print(f"Skipping {year} round {rnd}")

# Merge all race result frames into one modeling table.
df = pd.concat(results_list, ignore_index=True)

# Keep only columns used in this lab.
df = df[[
    'driverNumber',
    'givenName',
    'familyName',
    'constructorName',
    'grid',
    'position',
    'status',
    'points',
    'year',
    'round',
    'circuit'
]]

# Standardize column names for readability.
df.columns = [
    'DriverNumber',
    'FirstName',
    'LastName',
    'TeamName',
    'GridPosition',
    'Position',
    'Status',
    'Points',
    'Year',
    'Round',
    'Circuit'
]

# Build convenient identity and numeric fields.
df['FullName'] = df['FirstName'] + " " + df['LastName']
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# Feature engineering for baseline and analysis.
df['Top10'] = ((df['Position'] <= 10) & (df['Position'] >0)).astype(int)
df['BaselinePred'] = (df['GridPosition'] <= 10).astype(int)
df['PositionsGained'] = df['GridPosition'] - df['Position']
df['DNF'] = ~df['Status'].str.contains("Finished", na=False)

# Historical reliability and constructor strength proxies.
driver_reliability = 1 - df.groupby('FullName')['DNF'].mean()
df['DriverReliability'] = df['FullName'].map(driver_reliability)
constructor_points = df.groupby('TeamName')['Points'].mean()
df['ConstructorStrength'] = df['TeamName'].map(constructor_points)

# Remove unusable rows for grid/finish-based analyses.
df = df.dropna(subset=['GridPosition','Position'])

print("Dataset shape:", df.shape)


Request for URL https://api.jolpi.ca/ergast/f1/2023/8/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2023/8/results.json
Request for URL https://api.jolpi.ca/ergast/f1/2023/9/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\

Skipping 2023 round 11
Skipping 2023 round 12
Skipping 2023 round 13
Skipping 2023 round 14


Request for URL https://api.jolpi.ca/ergast/f1/2023/15/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2023/15/results.json


Skipping 2023 round 17
Skipping 2023 round 18
Skipping 2023 round 19
Skipping 2023 round 20
Skipping 2023 round 21
Skipping 2023 round 22
Skipping 2024 round 1
Skipping 2024 round 2


Request for URL https://api.jolpi.ca/ergast/f1/2024/3/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/3/results.json


Skipping 2024 round 4
Skipping 2024 round 6
Skipping 2024 round 7
Skipping 2024 round 8
Skipping 2024 round 9
Skipping 2024 round 10
Skipping 2024 round 11
Skipping 2024 round 12
Skipping 2024 round 13
Skipping 2024 round 14


Request for URL https://api.jolpi.ca/ergast/f1/2024/15/results.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests_cache\session.py", line 297, in _resend
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\gotts\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/15/results.json


Skipping 2024 round 16
Skipping 2024 round 18
Skipping 2024 round 19
Skipping 2024 round 20
Skipping 2024 round 21
Skipping 2024 round 22
Skipping 2024 round 23
Skipping 2024 round 24
Dataset shape: (1539, 18)


MARTIN!!! Do this splits for train/test/validate

In [2]:
df_trunq = df[['Top10','FullName','TeamName','GridPosition','Year','Circuit', 'Round']].copy()
df_trunq.head()
#splitting by Year, choice was for recency. 2023 to train (its a past season). 2024 to evaluate (test)
df_train = df_trunq[df_trunq["Year"].isin([2020,2021,2022])].copy()
df_val = df_trunq[df_trunq["Year"]==2023].copy()
df_test = df_trunq[df_trunq["Year"]==2024].copy()
print(f"Training shape: {df_train.shape}\nValidation shape: {df_val.shape}\nTesting shape: {df_test.shape}")

y_train = df_train["Top10"]
X_train = df_train.drop(["Top10",'Year'], axis=1)

y_val = df_val["Top10"] 
X_val = df_val.drop("Top10", axis=1)

y_test = df_test["Top10"] 
X_test = df_test.drop("Top10", axis=1)

Training shape: (1220, 7)
Validation shape: (240, 7)
Testing shape: (79, 7)


## Domain Heuristics

### 1. "Everyone that ends on top 10, ALWAYS ends in the top 10"

In [3]:
# Baseline 1: predict Top10 if the driver starts in the first 10 grid slots.


## Why this feature should help predict our target

### 2. "Driver rolling form (last 5 races) predicts Top-10"

In [ ]:
# Baseline 2: Use each driver's Top-10 rate from the previous 5 races.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_baseline(name, y_true, y_pred): # La podriamos mover arriba
    print(f"\n{name}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"F1       : {f1_score(y_true, y_pred, zero_division=0):.3f}")



df_form = df_trunq[['FullName', 'Year', 'Round', 'Top10']].copy()
df_form = df_form.sort_values(['FullName', 'Year', 'Round']).reset_index(drop=True)

df_form['DriverTop10RateLast5'] = (
    df_form.groupby('FullName')['Top10']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

# Global prior for first observed race of each driver.
global_top10_prior = df_train['Top10'].mean()
df_form['DriverTop10RateLast5'] = df_form['DriverTop10RateLast5'].fillna(global_top10_prior)

df_val_form = df_form[df_form['Year'] == 2023].copy()
df_test_form = df_form[df_form['Year'] == 2024].copy()

# Predict Top-10 if prior rolling probability >= 0.5.
pred_b2_val = (df_val_form['DriverTop10RateLast5'] >= 0.5).astype(int)
pred_b2_test = (df_test_form['DriverTop10RateLast5'] >= 0.5).astype(int)

evaluate_baseline('Baseline 2 (Validation): Driver rolling Top10 rate (last 5)', df_val_form['Top10'], pred_b2_val)
evaluate_baseline('Baseline 2 (Test): Driver rolling Top10 rate (last 5)', df_test_form['Top10'], pred_b2_test)


Baseline 2 (Validation): Driver rolling Top10 rate (last 5)
Accuracy : 0.721
Precision: 0.712
Recall   : 0.742
F1       : 0.727

Baseline 2 (Test): Driver rolling Top10 rate (last 5)
Accuracy : 0.810
Precision: 0.805
Recall   : 0.825
F1       : 0.815


## Why this feature should help predict our target

### 3. "Constructor historical Top-10 rate predicts Top-10"

In [5]:
# Baseline 3: Use constructor-level historical Top-10 probability from training data.
team_top10_rate = df_train.groupby('TeamName')['Top10'].mean()

df_val_b3 = df_val.copy()
df_test_b3 = df_test.copy()

df_val_b3['TeamTop10Rate'] = df_val_b3['TeamName'].map(team_top10_rate).fillna(global_top10_prior)
df_test_b3['TeamTop10Rate'] = df_test_b3['TeamName'].map(team_top10_rate).fillna(global_top10_prior)

pred_b3_val = (df_val_b3['TeamTop10Rate'] >= 0.5).astype(int)
pred_b3_test = (df_test_b3['TeamTop10Rate'] >= 0.5).astype(int)

evaluate_baseline('Baseline 3 (Validation): Constructor Top10 historical rate', y_val, pred_b3_val)
evaluate_baseline('Baseline 3 (Test): Constructor Top10 historical rate', y_test, pred_b3_test)


Baseline 3 (Validation): Constructor Top10 historical rate
Accuracy : 0.742
Precision: 0.742
Recall   : 0.742
F1       : 0.742

Baseline 3 (Test): Constructor Top10 historical rate
Accuracy : 0.519
Precision: 0.518
Recall   : 0.725
F1       : 0.604


## Why this feature should help predict our target

## Results/Comparissions